# 最終課題: 荷物の積付 — LLMプランナー(DSPy)と、GPUで学習した低レベル方策

**題材**: NEDO Challenge「Baggage-Loading Robot」の積付アルゴリズム(寸法や素材の異なる荷物を、航空コンテナへ安全に積む問題)を参考にした簡易版です。

**システム(2 層)**
```
指示 + 荷物の一覧 ─▶ [上位] LLMプランナー(DSPyで最適化) ─▶ pick / place_slot / place_on の列
                                                              │
荷物とアームの状態 ─▶ [下位]ニューラル方策 (GPUでDAgger学習) ─▶ 先端の移動量と吸着の開閉 ─▶ MuJoCo
```

| 層 | 最適化するもの | 報酬(評価の元) |
|---|---|---|
| 上位 | プロンプトとfew-shot例(DSPy MIPROv2) | 積付ルールを守って全荷物をコンテナに積めたか |
| 下位 | ニューラルネットの重み(PyTorch,GPU) | 熟練者の動作との一致 (模倣学習 + DAgger)。置く位置の許容誤差は8mmで、動作ノイズ(0.6cm)の下でも積み切れるか |

**荷物とコンテナ**: 荷物は7〜12個で、床スロット(6個)より多いので、必ず積み重ねます。大きさ(S/M/L)、重さ(重い/軽い)、割れ物かどうかが違います。
コンテナは床スロット6個で、最大2段まで積めます。12個のミッションは、アームの動作を実機に近い速さにしているため、2分以上かかります。

**積付ルール(プロンプトには書かず、失敗から学ばせる)**
| ルール | 違反したときのシミュレーターの反応 |
|---|---|
| 上に荷物が載っている荷物は持てない | エラー |
| 1 つの床スロットには1個だけ | エラー |
| 大きい荷物を小さい荷物の上に載せられない(L > M > S) | エラー(不安定) |
| 割れ物の上には何も載せられない | エラー |
| 重い荷物を軽い荷物の上に載せられない | エラー |
| 積めるのは床スロットの荷物の上だけ(最大2段) | エラー |

> 最適化されるのは上位の**プロンプト**と、下位の**ニューラルネットの重み**です。アームの機構(ガントリー+吸着)は自作で、実機の物理(把持の失敗や荷崩れ)は再現していません。レポートにもその旨を書いてください。

**実行環境**: ModalのGPUコンテナで、`run_v7_on_modal.py` がこのノートブックを先頭から最後まで自動実行します(手元: `modal run run_v7_on_modal.py`)。APIキーは ModalのSecret`anthropic-secret`(キー名 `ANTHROPIC_API_KEY`)に登録し、環境変数として渡されます。出力(結果ノートブック、動画、図、数値)はModal Volume`pai-final-outputs` に残り、実行後に手元の`modal_out/`へ自動でダウンロードされます。手順は`MODAL_README.md`にあります。

## 0. セットアップと設定

In [ ]:
# Modalのイメージ(run_v7_on_modal.py)には入っているので飛ばす。他の環境で動かすときはここで入れる
import os
if os.environ.get("ON_MODAL") != "1":
    get_ipython().system('pip -q install mujoco==3.17.0 "dspy[optuna]==3.4.1" mediapy opencv-python-headless')


In [ ]:
import os, sys, subprocess

def _gl_works(backend):
    """別プロセスで MUJOCO_GL=backend の描画コンテキストが作れるかを試す (import 後は切り替えられないため)"""
    code = "import os; os.environ['MUJOCO_GL']=%r; import mujoco; mujoco.GLContext(64, 64); print('ok')" % backend
    return subprocess.run([sys.executable, "-c", code], capture_output=True, timeout=120).returncode == 0

_gl = next((b for b in ("egl", "osmesa") if _gl_works(b)), None)               # ヘッドレス描画(mujocoをimportする前に設定)
if _gl is None:
    raise RuntimeError("EGL も OSMesa も使えません。イメージに libegl1 / libegl-mesa0 / libosmesa6 が入っているか確認してください")
os.environ["MUJOCO_GL"] = _gl
print("MUJOCO_GL =", os.environ["MUJOCO_GL"], "| ON_MODAL =", os.environ.get("ON_MODAL", "0"))

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("環境変数 ANTHROPIC_API_KEY がありません。Modal では Secret `anthropic-secret` (キー名 ANTHROPIC_API_KEY) を作り、"
                       "run_v7_on_modal.py から渡します。手元で動かすときは export ANTHROPIC_API_KEY=... としてください。")

import dspy

# ---- 実行時間と費用を決める設定 ----
N_EXPERT_EPISODES = 240        # 熟練者のデモの数 (1デモ = 荷物3個ぶんの動作)
DAGGER_ROUNDS, N_DAGGER_EPISODES = 4, 100
EPOCHS = 60
EVAL_MISSIONS = 24             # 低レベル方策の評価に使うミッション数 (少ないと成功率がばらつく)
N_TRAIN, N_DEV, N_TEST = 60, 40, 48   # プランナーの最適化・評価に使うミッション数
MIPRO_AUTO = "medium"          # "light" < "medium" < "heavy"
BUDGET_USD = 40.0              # 費用の目安の上限(LiteLLM が単価を知っているときだけ監視できる。ConsoleのLimitsも必ず設定する)

MODEL = "anthropic/claude-haiku-5-5"          # 実験用 (安く速い)
STRONG_MODEL = "anthropic/claude-sonnet-5-5"  # 指示文候補の生成と、モデル比較に使う
lm = dspy.LM(MODEL, max_tokens=4096)          # api_keyは環境変数から読まれる
strong_lm = dspy.LM(STRONG_MODEL, max_tokens=16000, cache=False)  # 4096だとSonnetは出力が途中で切れる/空になることがある
import time
from dspy.adapters.base import AdapterParseError

class RetryAdapter(dspy.ChatAdapter):
    """強いモデルの応答が解析できないとき (空応答や、途中で切れた JSON。Sonnet でまれに起きる) に、少し待ってやり直す。
    MIPROv2 の指示文提案は例外を捕まえないので、ここで吸収しないと実行全体が止まる (2026-11-01 の Modal 実行で 2 回発生)。
    キャッシュの効く実験用モデルは同じ応答が返るだけなので、再試行の対象にしない。"""
    def __init__(self, retry_models, tries=5, **kw):
        super().__init__(**kw)
        self.retry_models, self.tries, self.n_retry = set(retry_models), tries, 0

    def __call__(self, lm, lm_kwargs, signature, demos, inputs):
        for i in range(self.tries):
            try:
                return super().__call__(lm, lm_kwargs, signature, demos, inputs)
            except AdapterParseError as e:
                if lm.model not in self.retry_models or i == self.tries - 1:
                    raise
                self.n_retry += 1
                print(f"[RetryAdapter] {lm.model} の応答を解析できず -> {i + 1} 回目の再試行 | "
                      f"finish_reason={_last_finish_reason(lm)} | 応答の先頭: {str(getattr(e, 'lm_response', ''))[:80]!r}")
                time.sleep(2.0 * (i + 1))


def _last_finish_reason(lm):
    """直前の応答の finish_reason (途中で切れた理由の手がかり)。取れなければ None"""
    try:
        return lm.history[-1]["response"].choices[0].finish_reason
    except Exception:
        return None

adapter = RetryAdapter(retry_models={STRONG_MODEL})
dspy.configure(lm=lm, adapter=adapter, track_usage=True)
print(lm("Reply with the single word OK.")[0])   # 疎通確認 (キー・モデル名が正しいか)


def spent_usd():
    """LiteLLM が見積もった費用の合計。単価が不明なモデルでは 0 になるので、金額は Console の Usage を正とする。"""
    return sum(h["cost"] for m in (lm, strong_lm) for h in m.history if h.get("cost"))


def check_budget():
    s = spent_usd()
    print(f"calls: {len(lm.history)} (task) + {len(strong_lm.history)} (strong) | LiteLLM estimated cost: ${s:.2f} / ${BUDGET_USD:.0f} | 強いモデルの再試行 {adapter.n_retry} 回")
    assert s < BUDGET_USD, "費用の目安の上限を超えました。続ける前に Console の Usage を確認してください"

## 1. シミュレーター(MuJoCo): 荷物・コンテナ・アーム・積付ルール

In [ ]:
import os, re, random, time
import numpy as np
import mujoco

SHOW_RULES = False        # True: 場面記述に積付ルールを書く (False: ルールは失敗から学ばせる)
SIZES = {"S": (0.020, 0.020, 0.015), "M": (0.030, 0.025, 0.022), "L": (0.040, 0.030, 0.030)}   # 半辺 (x, y, z) [m]
RANK = {"S": 1, "M": 2, "L": 3}
MAX_ITEMS = 12
CONT, PITCH = np.array([0.30, 0.0]), 0.12                # コンテナ(床スロット3列x2行)の中心と、スロットの間隔
N_SLOTS = 6
SAFE_Z, GANTRY_Z0 = 0.24, 0.40
MAXD, SUBSTEPS = 0.02, 10                                # 1アクション(0.05秒)の最大移動2cm。実機に近い速さにして、作業を長くする
SKILLS = ["pick", "place_slot", "place_on"]
BASE_MASS = {True: 0.30, False: 0.05}                    # heavy / light [kg]
TOL_PLACE = 0.008          # 置いた位置の許容誤差 [m]。床は枠の中心から、積むときは下の荷物の中心から (外れると「落ちた/ずれた」とみなして失敗)


def slot_xy(k):
    return np.array([CONT[0] + ((k - 1) % 3 - 1) * PITCH, CONT[1] + ((k - 1) // 3 - 0.5) * PITCH])


def build_xml():
    bodies = ""
    for cls, (hx, hy, hz) in SIZES.items():
        for i in range(MAX_ITEMS):
            n = f"{cls}{i:02d}"
            bodies += f"""
    <body name="{n}" pos="0 0 {hz}"><freejoint name="{n}_j"/>
      <geom name="{n}_g" type="box" size="{hx} {hy} {hz}" rgba="0.6 0.6 0.6 1" mass="0.1" friction="1 0.01 0.001"/>
      <geom name="{n}_m" type="box" pos="0 0 {hz + 0.002}" size="{hx * 0.5} {hy * 0.5} 0.002" rgba="0.9 0.1 0.1 0" mass="0.001" contype="0" conaffinity="0"/></body>"""
    pads = "".join(
        f'<geom type="box" pos="{slot_xy(k)[0]} {slot_xy(k)[1]} 0.001" size="{PITCH/2-0.004} {PITCH/2-0.004} 0.001" rgba="{"0.32 0.36 0.45 1" if k % 2 else "0.40 0.44 0.55 1"}" contype="0" conaffinity="0"/>'
        for k in range(1, N_SLOTS + 1))
    walls = "".join(
        f'<geom type="box" pos="{CONT[0] + dx} {CONT[1] + dy}  0.02" size="{sx} {sy} 0.02" rgba="0.55 0.55 0.6 0.5" contype="0" conaffinity="0"/>'
        for dx, dy, sx, sy in ((0, -PITCH - 0.004, 1.5 * PITCH + 0.004, 0.004), (0, PITCH + 0.004, 1.5 * PITCH + 0.004, 0.004),
                               (-1.5 * PITCH - 0.004, 0, 0.004, PITCH), (1.5 * PITCH + 0.004, 0, 0.004, PITCH)))
    return f"""
<mujoco>
  <option timestep="0.005" gravity="0 0 -9.81"/>
  <visual><global offwidth="640" offheight="480"/><quality shadowsize="2048"/></visual>
  <asset>
    <texture name="grid" type="2d" builtin="checker" rgb1=".85 .85 .85" rgb2=".78 .78 .78" width="300" height="300"/>
    <material name="grid" texture="grid" texrepeat="8 6" reflectance="0.05"/>
  </asset>
  <worldbody>
    <light pos="0 -0.3 1.4" dir="0 0.2 -1" diffuse="0.9 0.9 0.9"/>
    <camera name="front" pos="0 -0.85 0.75" xyaxes="1 0 0 0 0.75 0.85"/>
    <geom name="floor" type="plane" size="0.7 0.5 0.1" material="grid"/>
    {pads}{walls}
    <body name="gantry" pos="0 0 {GANTRY_Z0}" gravcomp="1">
      <joint name="jx" type="slide" axis="1 0 0" range="-0.6 0.6" damping="45"/>
      <joint name="jy" type="slide" axis="0 1 0" range="-0.6 0.6" damping="45"/>
      <joint name="jz" type="slide" axis="0 0 1" range="-{GANTRY_Z0} 0.0" damping="45"/>
      <geom type="box" size="0.012 0.012 0.03" mass="1" rgba="0.15 0.15 0.15 1" contype="0" conaffinity="0"/>
      <geom type="cylinder" size="0.02 0.006" pos="0 0 -0.036" mass="0.1" rgba="0.9 0.5 0.1 1" contype="0" conaffinity="0"/>
      <site name="tip" pos="0 0 -0.042"/>
    </body>
    {bodies}
  </worldbody>
  <actuator>
    <position name="ax" joint="jx" kp="500"/><position name="ay" joint="jy" kp="500"/><position name="az" joint="jz" kp="500"/>
  </actuator>
</mujoco>"""


class World:
    """ガントリ型アームによる手荷物の積付。荷物は S/M/L の大きさ、重い/軽い、割れ物の属性を持つ。"""

    def __init__(self, render=False):
        self.model = mujoco.MjModel.from_xml_string(build_xml())
        self.data = mujoco.MjData(self.model)
        m = self.model
        self.body = {}                                         # 荷物名 -> body名(エピソードごとに割り当て)
        self.pool = {}
        for cls in SIZES:
            for i in range(MAX_ITEMS):
                n = f"{cls}{i:02d}"
                b = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, n)
                self.pool[n] = dict(bid=b, qadr=m.jnt_qposadr[m.body_jntadr[b]], dadr=m.jnt_dofadr[m.body_jntadr[b]],
                                    gid=mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, n + "_g"),
                                    mid=mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, n + "_m"),
                                    m0=float(m.body_mass[b]), I0=m.body_inertia[b].copy())
        self.tip = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_SITE, "tip")
        self.renderer = mujoco.Renderer(m, 480, 640) if render else None
        self.frames, self.recording, self.caption = [], False, ""
        self.items, self.held, self.last_held, self.log, self.act_noise = {}, None, None, [], 0.0
        self._rng = np.random.default_rng(0)
        self.frame_every, self._fc, self.sink = 1, 0, None   # 録画: Nアクションごとに1フレーム。sinkがあれば保存せず流す

    # ---------- 荷物の情報 ----------
    def hz(self, n):
        return SIZES[self.items[n]["size"]][2]

    def pos(self, n):
        return self.data.xpos[self.pool[self.body[n]]["bid"]].copy()

    def slot_of(self, n):
        p = self.pos(n)
        for k in range(1, N_SLOTS + 1):
            s = slot_xy(k)
            if abs(p[0] - s[0]) < PITCH / 2 and abs(p[1] - s[1]) < PITCH / 2 and p[2] < 0.16:
                return k
        return None

    def on_top_of(self, a, b):
        pa, pb = self.pos(a), self.pos(b)
        return abs(pa[0] - pb[0]) < 0.025 and abs(pa[1] - pb[1]) < 0.025 and abs((pa[2] - pb[2]) - (self.hz(a) + self.hz(b))) < 0.010

    def covered_by(self, n):
        return next((o for o in self.items if o != n and o != self.held and self.on_top_of(o, n)), None)

    def floor_occupant(self, k):
        return next((o for o in self.items if o != self.held and self.slot_of(o) == k and self.pos(o)[2] - self.hz(o) < 0.006), None)

    def describe(self):
        lines = []
        for n, it in self.items.items():
            if n == self.held:
                where = "held by the gripper"
            elif self.slot_of(n):
                under = next((o for o in self.items if o != n and self.on_top_of(n, o)), None)
                where = f"on top of {under}" if under else f"in container slot {self.slot_of(n)}"
            else:
                where = "in the staging area"
            d = SIZES[it["size"]]
            lines.append(f"- {n}: size={it['size']} ({d[0]*200:.0f}x{d[1]*200:.0f}x{d[2]*200:.1f}cm), "
                         f"weight={'heavy' if it['heavy'] else 'light'}, fragile={'yes' if it['fragile'] else 'no'}; {where}")
        lines.append(f"- container: {N_SLOTS} floor slots (numbered 1-{N_SLOTS}), at most 2 layers")
        if SHOW_RULES:
            lines += ["Loading rules:", "- a bag can only be picked up if nothing is on top of it",
                      "- each floor slot holds one bag; a bag can be stacked only on a bag that is on a floor slot",
                      "- a larger bag (L > M > S) cannot sit on a smaller bag", "- nothing may be placed on a fragile bag",
                      "- a heavy bag cannot sit on a light bag"]
        return "\n".join(lines)

    # ---------- 物理 ----------
    def _grab_sync(self):
        if self.held:
            p = self.pool[self.body[self.held]]
            self.data.qpos[p["qadr"]:p["qadr"] + 3] = self.tip_pos() - np.array([0, 0, self.hz(self.held) + 0.002])
            self.data.qpos[p["qadr"] + 3:p["qadr"] + 7] = [1, 0, 0, 0]
            self.data.qvel[p["dadr"]:p["dadr"] + 6] = 0

    def tip_pos(self):
        return self.data.site_xpos[self.tip].copy()

    def _step(self, n):
        for _ in range(n):
            mujoco.mj_step(self.model, self.data)
            self._grab_sync()

    def _frame(self):
        if self.recording and self.renderer is not None:
            self._fc += 1
            if self._fc % self.frame_every:
                return
            self.renderer.update_scene(self.data, camera="front")
            img = self.renderer.render()
            if self.sink:
                self.sink(img, self.caption)
            else:
                self.frames.append((img.copy(), self.caption))

    def reset(self, seed, items):
        """items: [dict(name, size, heavy, fragile), ...]。荷物は左側の仮置き場にランダム配置。"""
        rng = random.Random(seed)
        m, d = self.model, self.data
        mujoco.mj_resetData(m, d)
        self.items = {it["name"]: it for it in items}
        self.body, self.held, self.last_held, self.log, self.frames = {}, None, None, [], []
        used = {c: 0 for c in SIZES}
        for j, p in enumerate(self.pool.values()):                       # 使わない荷物は視野の外に退避
            d.qpos[p["qadr"]:p["qadr"] + 7] = [1.0 + 0.12 * (j % 6), 1.0 + 0.12 * (j // 6), 0.03, 1, 0, 0, 0]
            m.geom_rgba[p["mid"]][3] = 0.0
        pts = []
        for it in items:
            cls = it["size"]
            bn = f"{cls}{used[cls]:02d}"; used[cls] += 1
            self.body[it["name"]] = bn
            p = self.pool[bn]
            mass = BASE_MASS[it["heavy"]]
            m.body_mass[p["bid"]] = mass
            m.body_inertia[p["bid"]] = p["I0"] * mass / p["m0"]
            base = np.array([0.35, 0.45, 0.75]) if cls == "S" else (np.array([0.35, 0.6, 0.35]) if cls == "M" else np.array([0.85, 0.55, 0.25]))
            m.geom_rgba[p["gid"]] = np.r_[base * (0.45 if it["heavy"] else 1.0), 1.0]
            m.geom_rgba[p["mid"]][3] = 1.0 if it["fragile"] else 0.0
            while True:
                q = (rng.uniform(-0.52, 0.02), rng.uniform(-0.26, 0.26))
                if all(np.hypot(q[0] - r[0], q[1] - r[1]) > 0.095 for r in pts):
                    pts.append(q); break
            d.qpos[p["qadr"]:p["qadr"] + 7] = [q[0], q[1], SIZES[cls][2], 1, 0, 0, 0]
        d.qpos[:3] = 0
        d.ctrl[:3] = 0
        mujoco.mj_forward(m, d)
        self._move_direct(-0.50, -0.40, SAFE_Z)
        self._step(100)

    def _move_direct(self, x, y, z, steps=1200):
        self.data.ctrl[:3] = self.data.qpos[:3] + (np.array([x, y, z]) - self.tip_pos())
        self._step(steps)

    # ---------- スキルの目標とルール ----------
    def skill_goal(self, skill, arg):
        if skill == "pick":
            if self.held:
                raise RuntimeError(f"the gripper is already holding {self.held}")
            if arg not in self.items:
                raise RuntimeError(f"unknown bag '{arg}'")
            cover = self.covered_by(arg)
            if cover:
                raise RuntimeError(f"cannot pick {arg}: {cover} is on top of it")
            p = self.pos(arg)
            return dict(skill=0, name="pick", target=arg, g=np.array([p[0], p[1], p[2] + self.hz(arg) + 0.002]))
        if not self.held:
            raise RuntimeError("the gripper is not holding anything")
        h = self.held
        if skill == "place_slot":
            try:
                k = int(arg)
            except (TypeError, ValueError):
                raise RuntimeError(f"unknown slot '{arg}'")
            if not 1 <= k <= N_SLOTS:
                raise RuntimeError(f"slot {k} does not exist (slots are 1-{N_SLOTS})")
            occ = self.floor_occupant(k)
            if occ:
                raise RuntimeError(f"slot {k} is already occupied by {occ}")
            s = slot_xy(k)
            return dict(skill=1, name="place_slot", target=k, g=np.array([s[0], s[1], 2 * self.hz(h) + 0.005]))
        # place_on
        if arg not in self.items or arg == h:
            raise RuntimeError(f"cannot place {h} on '{arg}'")
        t = self.items[arg]; me = self.items[h]
        if not self.slot_of(arg):
            raise RuntimeError(f"cannot stack on {arg}: stacking is only allowed inside the container")
        if self.pos(arg)[2] - self.hz(arg) > 0.006:
            raise RuntimeError(f"cannot stack on {arg}: at most 2 layers are allowed")
        if self.covered_by(arg):
            raise RuntimeError(f"{arg} already has a bag on top of it")
        if RANK[me["size"]] > RANK[t["size"]]:
            raise RuntimeError(f"unstable: {h} ({me['size']}) is larger than {arg} ({t['size']})")
        if t["fragile"]:
            raise RuntimeError(f"{arg} is fragile: nothing may be placed on it")
        if me["heavy"] and not t["heavy"]:
            raise RuntimeError(f"{h} is heavy and {arg} is light: a heavy bag cannot sit on a light bag")
        p = self.pos(arg)
        return dict(skill=2, name="place_on", target=arg, g=np.array([p[0], p[1], p[2] + self.hz(arg) + 2 * self.hz(h) + 0.005]))

    def obs(self, goal):
        ee = self.tip_pos()
        oh = np.zeros(3); oh[goal["skill"]] = 1
        return np.concatenate([ee, [float(self.held is not None)], goal["g"], goal["g"] - ee, oh]).astype(np.float32)   # 13次元

    def apply(self, delta, grip, goal):
        """1アクション(0.05秒)。delta は先端の移動量(最大 MAXD)、grip>0.5 で吸着、<=0.5 で解放。"""
        delta = np.asarray(delta, dtype=np.float64)
        if self.act_noise:
            delta = delta + self.act_noise * self._rng.standard_normal(3)
        delta = np.clip(delta, -MAXD, MAXD)
        tip = self.tip_pos()
        self.data.ctrl[:3] = self.data.qpos[:3] + delta
        self._step(SUBSTEPS)
        if grip > 0.5 and self.held is None and goal["skill"] == 0:
            e = self.tip_pos() - goal["g"]
            if np.hypot(e[0], e[1]) < 0.012 and abs(e[2]) < 0.010:
                self.held = goal["target"]
        elif grip <= 0.5 and self.held is not None:
            self.last_held, self.held = self.held, None
            for _ in range(4):
                self._step(10); self._frame()
        self._frame()

    def skill_done(self, goal):
        up = self.tip_pos()[2] >= SAFE_Z - 0.02
        return up and ((self.held == goal["target"]) if goal["skill"] == 0 else (self.held is None))

    def skill_effect_ok(self, goal):
        c = self.last_held
        if goal["skill"] == 1:
            s = slot_xy(goal["target"])
            return (self.slot_of(c) == goal["target"] and self.pos(c)[2] - self.hz(c) < 0.008
                    and np.hypot(*(self.pos(c)[:2] - s)) < TOL_PLACE)
        t = self.pos(goal["target"])
        return self.on_top_of(c, goal["target"]) and np.hypot(*(self.pos(c)[:2] - t[:2])) < TOL_PLACE

## 2. 熟練者コントローラ、スキルの実行、積付ソルバー
- `expert_action`: 観測から「先端の移動量」と「吸着の開閉」を返す手書きの熟練者。下位方策の教師になります。
- `solve_loading`: ルールを知っているソルバー。学習データの生成と動作確認にだけ使い、LLM には渡しません。
- `run_skill_fast`: アームの動作を省き、スキルの結果だけを反映する高速実行です。プランナーの採点(数千回)に使います。

In [ ]:
# ---------- 熟練者(スクリプト)コントローラ: 観測 -> (先端の移動量, grip) ----------
def expert_action(o):
    ee, held, g = o[0:3], o[3], o[4:7]
    skill = int(np.argmax(o[10:13]))
    is_pick = skill == 0
    moving = (held < 0.5) if is_pick else (held > 0.5)      # まだ目標へ向かう段階か
    grip = float(held > 0.5)
    if moving:
        xy = np.hypot(g[0] - ee[0], g[1] - ee[1])
        thr = 0.012 if ee[2] < SAFE_Z - 0.03 else 0.006      # 降下中は少し広い許容 (ヒステリシス)
        if xy > thr:
            if ee[2] < SAFE_Z - 0.03 and xy > 0.03:
                tgt = np.array([ee[0], ee[1], SAFE_Z])      # まず真上へ戻る
            else:
                tgt = np.array([g[0], g[1], SAFE_Z])         # 安全な高さで水平移動
        else:
            tgt = g.copy()                                   # 目標の真上: 降下
            if abs(ee[2] - g[2]) < 0.006:
                grip = 1.0 if is_pick else 0.0               # 吸着 / 解放
    else:
        tgt = np.array([ee[0], ee[1], SAFE_Z])               # 作業後は真上へ戻る
    return np.clip(tgt - ee, -MAXD, MAXD), grip


# ---------- スキルとプランの実行 ----------
SKILL_RE = re.compile(r"^(pick|place_slot|place_on)\s*\(\s*['\"]?([a-z0-9_]*)['\"]?\s*\)$")


def parse_plan(text):
    """LLM の出力の書式ゆれ(コードフェンス・番号・引用符・大文字・; 区切り)を許容して解析する。"""
    steps = []
    for raw in text.strip().splitlines():
        line = raw.strip()
        if not line or line.startswith("```") or line.startswith("#"):
            continue
        line = re.sub(r"^\s*(\d+[.)]|[-*])\s*", "", line)
        for part in filter(None, (p.strip() for p in line.split(";"))):
            m = SKILL_RE.match(part.lower())
            if not m:
                raise ValueError(f"cannot parse: {part!r}")
            steps.append((m.group(1), m.group(2) or None))
    return steps


def run_skill_fast(w, skill, arg):
    """アームの動作を省き、スキルの結果だけを反映する(ルール判定と、置いたあとの物理は同じ)。プランナーの採点を速くするために使う。"""
    goal = w.skill_goal(skill, arg)
    if goal["skill"] == 0:
        w.held = goal["target"]
        return
    h = w.held
    p = w.pool[w.body[h]]
    x, y, z = goal["g"]
    w.data.qpos[p["qadr"]:p["qadr"] + 7] = [x, y, z - w.hz(h) - 0.002, 1, 0, 0, 0]
    w.data.qvel[p["dadr"]:p["dadr"] + 6] = 0
    w.last_held, w.held = h, None
    w._step(60)
    if not w.skill_effect_ok(goal):
        raise RuntimeError(f"{w.last_held} did not end up at the target ({skill} {arg})")


def run_skill(w, skill, arg, controller, max_actions=300):
    if controller == "fast":
        return run_skill_fast(w, skill, arg)
    goal = w.skill_goal(skill, arg)
    for _ in range(max_actions):
        d, grip = controller(w.obs(goal))
        w.apply(d, grip, goal)
        if w.skill_done(goal):
            if goal["skill"] != 0 and not w.skill_effect_ok(goal):
                raise RuntimeError(f"{w.last_held} did not end up at the target ({skill} {arg})")
            return
    raise RuntimeError(f"{skill}({arg or ''}) timed out")


def execute(w, plan_text, controller, max_skills=60):
    """成功なら (True, ''), 失敗なら (False, 理由)。"""
    try:
        steps = parse_plan(plan_text)
        if not steps:
            return False, "empty plan"
        if len(steps) > max_skills:
            return False, "plan too long"
        for skill, arg in steps:
            w.caption = f"{skill}({arg or ''})"
            w.log.append(w.caption)
            run_skill(w, skill, arg, controller)
        if w.held:
            return False, f"the plan ended while holding {w.held}"
        w.caption = ""
        return True, ""
    except (RuntimeError, ValueError) as e:
        return False, str(e)


# ---------- 積付の正解(ルールを知っているソルバー。学習データと動作確認に使い、LLMには渡さない) ----------
def can_stack(top, bot):
    return (RANK[top["size"]] <= RANK[bot["size"]] and not bot["fragile"] and not (top["heavy"] and not bot["heavy"]))


def solve_loading(items):
    """items: {名前: 属性}。大きく重い荷物から順に、床スロットか、床の荷物の上へ置く割り当てを探索する。"""
    names = sorted(items, key=lambda n: (-RANK[items[n]["size"]], not items[n]["heavy"], items[n]["fragile"], n))
    floors, tops, plan = [], {}, []

    def rec(i):
        if i == len(names):
            return True
        n = names[i]
        if len(floors) < N_SLOTS:
            floors.append(n); plan.append((n, "place_slot", len(floors)))
            if rec(i + 1):
                return True
            floors.pop(); plan.pop()
        for b in floors:
            if b not in tops and can_stack(items[n], items[b]):
                tops[b] = n; plan.append((n, "place_on", b))
                if rec(i + 1):
                    return True
                del tops[b]; plan.pop()
        return False

    return plan if rec(0) else None


ORDER_RULES = {
    "fragile_last": ("割れ物の荷物は、ほかの荷物がすべて運び終わってから最後に扱う", lambda it: it["fragile"]),
    "heavy_first": ("重い荷物は、軽い荷物より先に運ぶ", lambda it: not it["heavy"]),
}


def oracle_plan(task):
    """正解の計画。指示に順序の要求があるときは、置き場所の割り当てを変えずに、要求を満たす順に並べ替える。
    (土台は上に載せる荷物より先に置く、という依存関係は、同じ群の中で元の順序を保つので壊れない)"""
    items = {it["name"]: it for it in task["items"]}
    plan = solve_loading(items)
    rule = task.get("order")
    if rule:
        late = ORDER_RULES[rule][1]
        plan = sorted(plan, key=lambda step: bool(late(items[step[0]])))      # 安定ソート:後ろに回す群だけを後ろへ
    return "\n".join(f"pick({n})\n{sk}({arg})" for n, sk, arg in plan)


def order_ok(task, w):
    """実際に持ち上げた順序が、指示の要求を満たしたか。(True, '') か (False, 理由)。"""
    rule = task.get("order")
    if not rule:
        return True, ""
    late = ORDER_RULES[rule][1]
    picks = [c[5:-1] for c in w.log if c.startswith("pick(")]
    attr = {it["name"]: it for it in task["items"]}
    seq = [bool(late(attr[n])) for n in picks if n in attr]
    if any(a and not b for a, b in zip(seq, seq[1:])):
        return False, "the order request in the instruction was not followed"
    return True, ""

## 3. ミッションの生成と成否判定
ミッションは4種類をseedで循環させます(`stack` 7〜8 個 / `mixed` 9〜10 個 / `dense` 10〜11 個 / `tight` 12 個)。どの種類も床スロット(6 個)を超えるので、積み重ねが必須です。
さらに、半数のミッションには、指示文に順序の要求が付きます(英語と日本語が混ざります): 「割れ物は最後に扱う」(`fragile_last`)、「重い荷物から先に運ぶ」(`heavy_first`)。成功には、荷物を積み切ることに加えて、実際に持ち上げた順序が要求どおりであることが必要です。荷物の一覧だけを見て指示文を読まない計画は、失敗します。どれも、ルールを守れば必ず積み切れる荷物の組です。

In [ ]:
def make_task(seed):
    """seed から、荷物の組(必ず積める)と指示文を決める。成否は最終的な荷物の位置だけで判定する。"""
    rng = random.Random(30_000 + seed)
    kind = ["stack", "mixed", "dense", "tight"][seed % 4]      # どの種類も、床スロット 6 個を超えるので積み重ねが必須
    n = {"stack": rng.choice([7, 8]), "mixed": rng.choice([9, 10]), "dense": rng.choice([10, 11]), "tight": 12}[kind]
    n_bottom = min(n, N_SLOTS); n_top = n - n_bottom
    bottoms, tops = [], []
    for i in range(n_bottom):
        holds = i < n_top                                           # 上に1個載せる床の荷物は、割れ物にしない
        bottoms.append(dict(size=rng.choice(["M", "L", "L"]) if holds else rng.choice("SML"),
                            heavy=rng.random() < 0.5, fragile=(not holds) and rng.random() < 0.4))
    for i in range(n_top):
        b = bottoms[i]
        size = rng.choice([s for s in "SML" if RANK[s] <= RANK[b["size"]]])
        tops.append(dict(size=size, heavy=b["heavy"] and rng.random() < 0.5, fragile=rng.random() < 0.4))
    items = bottoms + tops
    rng.shuffle(items)
    for i, it in enumerate(items):
        it["name"] = f"bag{i + 1:02d}"
    assert solve_loading({it["name"]: it for it in items}) is not None
    orng = random.Random(70_000 + seed)                       # 荷物の生成とは別の乱数 (荷物の組は v5 と同じ)
    order = orng.choice([None, None, "fragile_last", "heavy_first"])
    text = {
        None: [f"Load all {n} bags into the container."],
        "fragile_last": [f"Load all {n} bags into the container. Handle the fragile bags last.",
                         f"コンテナに荷物 {n} 個をすべて積んでください。割れ物は最後に扱ってください。"],
        "heavy_first": [f"Load all {n} bags into the container. Move the heavy bags first.",
                        f"コンテナに荷物 {n} 個をすべて積んでください。重い荷物から先に運んでください。"],
    }[order]
    return dict(seed=seed, kind=kind, n=n, items=items, order=order, instruction=orng.choice(text))


def goal_reached(w):
    return all(w.slot_of(n) is not None for n in w.items)


import threading
_TLS = threading.local()


def _cached_world():
    """スレッドごとに World を 1 つ使い回す (モデルのコンパイルを毎回しないため)。"""
    if not hasattr(_TLS, "w"):
        _TLS.w = World()
    return _TLS.w


def run_task(task, plan_text, controller=None, world=None, record=False, act_noise=0.0):
    """controller: expert_action / NNController / "fast" (動作を省いた高速実行。プランナーの採点用)。"""
    w = world or (World(render=True) if record else _cached_world())
    w.act_noise = act_noise
    w.reset(task["seed"], task["items"])
    w.recording = record
    ok, err = execute(w, plan_text, controller or expert_action)
    w._step(60)
    w.recording = False
    w.act_noise = 0.0
    if ok and goal_reached(w):
        ok, err = order_ok(task, w)                        # 指示の順序の要求を守ったか
        return ok, err, w
    return False, err, w


def task_scene(task):
    w = _cached_world(); w.reset(task["seed"], task["items"])
    return w.describe()

## 4. 動作確認(LLM も学習もまだ使わない)

In [ ]:
import mediapy, numpy as np

t = make_task(0)
print(t["instruction"]); print(task_scene(t)); print()
plan = oracle_plan(t); print(plan.replace("\n", " | "))

REC = World(render=True)                      # 録画用のWorldは1つを使い回す
REC.frame_every = 2                           # 2アクションごとに1フレーム(20fpsで書き出すと2倍速)
ok, err, w = run_task(t, plan, world=REC, record=True)
print("success:", ok, err, f"| sim time {w.data.time:.0f}s")
mediapy.show_video([f for f, _ in w.frames], fps=20, width=480, title="熟練者コントローラ (2 倍速)")

In [ ]:
# 高速実行が本実行と同じ判定になることの確認 (正解プランを壊したプランで比べる)
import random as _r

def mutate(plan, rng):
    L = plan.splitlines(); pairs = [(L[i], L[i + 1]) for i in range(0, len(L), 2)]
    m = rng.choice(["swap", "drop", "slot", "onto", "none"])
    if m == "swap" and len(pairs) > 1:
        i, j = rng.sample(range(len(pairs)), 2); pairs[i], pairs[j] = pairs[j], pairs[i]
    elif m == "drop":
        pairs.pop(rng.randrange(len(pairs)))
    elif m == "slot":
        i = rng.randrange(len(pairs)); pairs[i] = (pairs[i][0], f"place_slot({rng.randint(1, 7)})")
    elif m == "onto":
        i = rng.randrange(len(pairs)); pairs[i] = (pairs[i][0], f"place_on(bag{rng.randint(1, 12):02d})")
    return "\n".join(a + "\n" + b for a, b in pairs)

rng, agree, N = _r.Random(0), 0, 24
for s in range(40, 40 + N):
    tk = make_task(s); pl = mutate(oracle_plan(tk), rng)
    ok_f, err_f, _ = run_task(tk, pl, controller="fast")
    ok_e, err_e, _ = run_task(tk, pl, controller=expert_action)
    agree += (ok_f == ok_e) and (ok_e or err_f.split()[:3] == err_e.split()[:3])
print(f"fast と本実行の判定が一致: {agree} / {N}")

## 5. 下位方策をGPUで学習する(模倣学習 + DAgger)
1. 熟練者の動作を、荷物3個ぶんの区間ごとに集める(区間より前は高速実行で進め、積み上げ済みの場面も均等に集める)。
2. 全結合ネットワーク(13次元の観測 → 先端の移動量3 + 吸着の開閉1)を**GPUで**学習する。
3. 学習した方策で動かし、**方策が実際に訪れた状態**に熟練者のラベルを付けて集め直す(DAgger)。これを繰り返す。
4. 毎ラウンド、正解プランを実行してミッションの成功率を測る。

**この課題でDAggerの効果が出る理由**: 置く位置の許容誤差は8mmと厳しく、実行時には毎ステップ動作ノイズ(0.6cm)が加わります。
熟練者のデモはノイズなしの理想的な軌道だけなので、そのまま模倣しただけの方策(ラウンド 0 = 通常の模倣学習)は、ノイズで軌道を外れたあとの状態を一度も見ておらず、立て直せません。
DAggerは、方策がノイズの下で実際に訪れた状態に熟練者のラベルを付け直すので、立て直し方を学べます。

In [ ]:
import copy
import torch, torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(GPU が無いので学習は遅くなります)")


def featurize(o):
    x = np.array(o, dtype=np.float32, copy=True)
    x[..., 0:3] *= 4; x[..., 4:7] *= 4; x[..., 7:10] *= 40     # 座標は1前後、相対位置はmmの差が見える大きさに
    return x


class PolicyNet(nn.Module):
    """観測(13次元) -> [先端の移動量3(MAXD で正規化),gripのロジット1]。"""
    def __init__(self, hidden=512, depth=4):
        super().__init__()
        layers, d = [], 13
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.LayerNorm(hidden), nn.GELU()]
            d = hidden
        self.body, self.head = nn.Sequential(*layers), nn.Linear(hidden, 4)

    def forward(self, x):
        return self.head(self.body(x))


class NNController:
    """学習済みネットワークを、シミュレーターのループで使うためのラッパー(推論はCPUのコピーで行う)。"""
    def __init__(self, net):
        self.net = net
        self.sync()

    def sync(self):
        self.cpu = copy.deepcopy(self.net).cpu().eval()

    def __call__(self, o):
        with torch.no_grad():
            out = self.cpu(torch.from_numpy(featurize(o))[None])[0].numpy()
        return out[:3] * MAXD, float(out[3] > 0)


def collect(n_episodes, seed0, exec_ctrl=None, beta=1.0, noise=0.0, window=3, act_noise=0.0):
    """ミッションの途中から荷物window個ぶんの動作を実行し、毎ステップ熟練者のラベルを記録する。
    それより前の動作は高速実行で済ませる(積み上げ済みの状態など、後半の場面も均等に集めるため)。
    実行はbetaの確率で熟練者、それ以外はexec_ctrl(DAgger)。"""
    X, Y, w, rng = [], [], World(), np.random.default_rng(seed0)
    for e in range(n_episodes):
        task = make_task(seed0 + e)
        steps = parse_plan(oracle_plan(task))
        w.reset(task["seed"], task["items"])
        w.act_noise = act_noise
        i0 = int(rng.integers(0, len(steps) // 2)) * 2
        try:
            for skill, arg in steps[:i0]:
                run_skill_fast(w, skill, arg)
        except RuntimeError:
            continue
        for skill, arg in steps[i0:i0 + 2 * window]:
            try:
                goal = w.skill_goal(skill, arg)
            except RuntimeError:
                break
            done = False
            for _ in range(300):
                o = w.obs(goal)
                d_e, g_e = expert_action(o)
                X.append(featurize(o)); Y.append(np.r_[d_e / MAXD, g_e])
                if exec_ctrl is None or rng.random() < beta:
                    d, g = d_e + noise * rng.standard_normal(3), g_e
                else:
                    d, g = exec_ctrl(o)
                w.apply(d, g, goal)
                if w.skill_done(goal):
                    done = True; break
            if not done:
                break
    return np.array(X, np.float32), np.array(Y, np.float32)


def train(net, X, Y, epochs=30, lr=1e-3, bs=2048):
    """GPU上で模倣学習 (移動量はMSE、gripはBCE)。"""
    net.to(DEVICE).train()
    Xt, Yt = torch.from_numpy(X).to(DEVICE), torch.from_numpy(Y).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    steps = epochs * ((len(Xt) + bs - 1) // bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    bce = nn.BCEWithLogitsLoss()
    t0 = time.time()
    for ep in range(epochs):
        perm = torch.randperm(len(Xt), device=DEVICE)
        tot = 0.0
        for i in range(0, len(Xt), bs):
            idx = perm[i:i + bs]
            out = net(Xt[idx])
            loss = nn.functional.mse_loss(out[:, :3], Yt[idx, :3]) * 20 + bce(out[:, 3], Yt[idx, 3])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            tot += loss.item() * len(idx)
        if ep % 10 == 0 or ep == epochs - 1:
            print(f"  epoch {ep:3d}  loss {tot / len(Xt):.4f}")
    print(f"  学習時間 {time.time() - t0:.1f}s ({DEVICE}, {len(Xt)} samples)")
    net.eval()
    return net


def eval_controller(ctrl, seeds, act_noise=0.0):
    """正解プランを、与えたコントローラで実行したときの成功率 (計画の誤りを混ぜず、低レベル方策だけを測る)。
    act_noiseは実行時に加える動作ノイズ[m]。現実の外乱を模した、ロバスト性の評価に使う。"""
    plan_ok, skills_ok, skills_all = 0, 0, 0
    for s in seeds:
        task = make_task(s); plan = oracle_plan(task)
        n = len(plan.splitlines())
        ok, err, w = run_task(task, plan, controller=ctrl, act_noise=act_noise)
        plan_ok += ok
        skills_all += n
        skills_ok += n if ok else max(len(w.log) - 1, 0)       # 失敗したスキルの手前まで成功とみなす
    return plan_ok / len(seeds), skills_ok / skills_all

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import json

eval_seeds = list(range(5000, 5000 + EVAL_MISSIONS))
history = []
t_all = time.time()

print("== 熟練者(基準)==")
ref_clean = eval_controller(expert_action, eval_seeds)
ref_noisy = eval_controller(expert_action, eval_seeds, 0.006)
print(f"expert  | ミッション成功率 clean {ref_clean[0]:.0%}, ノイズ {ref_noisy[0]:.0%}   (ノイズ下でも 100% にならないのは、許容誤差 8mm が厳しいため)")

print("\n== ラウンド 0: 熟練者のデモだけで学習(通常の模倣学習。デモにノイズは入れない)==")
X, Y = collect(N_EXPERT_EPISODES, 10000, exec_ctrl=None, beta=1.0, noise=0.0)       # seed 10000〜:  train/dev/test(0〜,1000〜,2000〜)と重ねない
print("samples:", X.shape)
policy = train(PolicyNet(), X, Y, epochs=EPOCHS)
ctrl = NNController(policy)
bc_ctrl = NNController(policy)               # ラウンド0の方策を、比較用にこの時点で固定して残す(重みのCPUコピー)


def log_round(name):
    c, n = eval_controller(ctrl, eval_seeds), eval_controller(ctrl, eval_seeds, 0.006)
    history.append(dict(round=name, samples=len(X), plan_clean=c[0], skill_clean=c[1], plan_noisy=n[0], skill_noisy=n[1]))
    print(f"round {name} | samples {len(X):7d} | ミッション成功率 clean {c[0]:.0%} / ノイズ {n[0]:.0%} | スキル成功率 clean {c[1]:.1%} / ノイズ {n[1]:.1%}")


log_round(0)
for r in range(1, DAGGER_ROUNDS + 1):
    print(f"\n== ラウンド {r}: 学習した方策で動かし、熟練者のラベルを付けて追加(DAgger)==")
    X2, Y2 = collect(N_DAGGER_EPISODES, 20000 + 1000 * r, exec_ctrl=ctrl, beta=max(0.4 - 0.2 * r, 0.0), act_noise=0.006)   # 実行時の条件(動作ノイズ)の下で集める
    X, Y = np.concatenate([X, X2]), np.concatenate([Y, Y2])
    policy = train(policy, X, Y, epochs=EPOCHS)
    ctrl.sync()
    log_round(r)

torch.save(policy.state_dict(), "policy.pt")
print(f"\n下位方策の学習と評価の合計: {(time.time() - t_all) / 60:.1f} 分 | device: {DEVICE}")

fig, ax = plt.subplots(figsize=(6.4, 4.0), dpi=110)
xs = [h["round"] for h in history]
ax.plot(xs, [h["plan_clean"] * 100 for h in history], "o-", label="mission success (clean)")
ax.plot(xs, [h["plan_noisy"] * 100 for h in history], "s-", label="mission success (0.6cm noise)")
ax.set_xticks(xs); ax.set_xticklabels(["BC" if x == 0 else str(x) for x in xs])
ax.axhline(ref_noisy[0] * 100, ls="--", c="gray", label="scripted expert (0.6cm noise)")
ax.set_xlabel("round (0 = behavior cloning only, 1+ = DAgger)"); ax.set_ylabel("%"); ax.set_ylim(-2, 105); ax.legend(loc="lower right"); fig.tight_layout()
fig.savefig("dagger_curve.png"); plt.show()
# セル41まで届かずに止まっても、ここまでの数値がVolumeに残るようにする
import json
json.dump(dict(policy_history=history, expert_clean=ref_clean, expert_noise=ref_noisy, eval_missions=EVAL_MISSIONS, device=DEVICE),
          open("policy_history.json", "w"), default=float)
print("policy_history.json に保存")

## 6. 上位プランナー: データセットとDSPyモジュール
`train` で最適化し、`dev` で最適化中の候補を選び、**`test` は最終比較にだけ使います**。
プランナーの採点には高速実行(`controller="fast"`)を使います。低レベル方策の誤りが混ざらないので、「計画の良し悪し」だけを測れます。

**場面記述の切り替え**: `SHOW_RULES = False` のとき、ルールは書かれていません。次節のベースラインが極端に低い(目安 20% 未満)ときは、`True` にして、このセルから再実行してください。

In [ ]:
SHOW_RULES = False       # 次節のベースラインが20%未満ならTrueにして、このセル以降を再実行

TASK_BY_SEED = {}        # 正解(荷物の属性)はここに置き、dspy.Exampleには渡さない

def to_example(seed):
    t = make_task(seed)
    TASK_BY_SEED[seed] = t
    return dspy.Example(instruction=t["instruction"], scene=task_scene(t), seed=seed).with_inputs("instruction", "scene")

trainset = [to_example(s) for s in range(0, N_TRAIN)]
devset   = [to_example(s) for s in range(1000, 1000 + N_DEV)]
testset  = [to_example(s) for s in range(2000, 2000 + N_TEST)]
print(len(trainset), len(devset), len(testset))
print(testset[3].instruction); print(testset[3].scene)

In [ ]:
class Plan(dspy.Signature):
    """Write a loading plan for a robot arm as a list of skill calls, one per line.
    The instruction may contain a request about the order of handling (in English or Japanese); follow it.
    Skills: pick(bag), place_slot(slot_number), place_on(bag)."""
    instruction: str = dspy.InputField()
    scene: str = dspy.InputField(desc="attributes and locations of the bags and the container")
    plan: str = dspy.OutputField(desc="one skill call per line, e.g. pick(bag01) then place_slot(1)")


class Planner(dspy.Module):
    def __init__(self):
        super().__init__()
        self.plan = dspy.ChainOfThought(Plan)

    def forward(self, instruction, scene):
        return self.plan(instruction=instruction, scene=scene)


def metric(example, pred, trace=None):
    """シミュレーターで実行して、全荷物をコンテナに積めたら 1 (高速実行)。"""
    ok, _, _ = run_task(TASK_BY_SEED[example.seed], getattr(pred, "plan", None) or "", controller="fast")
    return float(ok)


evaluate = dspy.Evaluate(devset=testset, metric=metric, num_threads=6, display_progress=True, display_table=0)

## 7. 最適化前(ベースライン)の成功率

In [ ]:
baseline = Planner()
base_res = evaluate(baseline)
print("baseline success rate:", base_res.score, "%")
check_budget()

**判断ゲート(ここで必ず確認)**

| ベースライン | 次の行動 |
|---|---|
| 20% 未満 | ルールが見えず、最適化が手探りになります。6 章で `SHOW_RULES = True` にして、6 章から再実行 |
| 20〜85% | そのまま 8 章へ |
| 85% 以上 | 対比が出ません。`N_TEST` や荷物数を増やす(`tight` の比率を上げる)か、`MODEL` をより小さいモデルにする |

## 8. 最適化(MIPROv2)
- 指示文の候補を`STRONG_MODEL`が作り、成功した実行例をfew-shotとして選び、`dev` での成功率が高い組み合わせを探します。
- `MIPRO_AUTO = "medium"`で、呼び出し回数は千回前後です。実行前にConsoleのUsage とLimits(月の上限)を確認してください。

In [ ]:
from dspy.teleprompt import MIPROv2

def compile_planner(prompt_model):
    tp = MIPROv2(metric=metric, prompt_model=prompt_model, task_model=lm, auto=MIPRO_AUTO,
                 num_threads=6, max_bootstrapped_demos=4, max_labeled_demos=0, seed=0)
    return tp.compile(Planner(), trainset=trainset, valset=devset,
                      minibatch_size=len(devset),   # 既定の 35 が valset より大きいと ValueError
                      requires_permission_to_run=False)

# 指示文の提案は強いモデル(Sonnet)に任せる。再試行しても応答を解析できないときは、
# 50分ぶんの下位方策の学習を無駄にしないため、提案モデルを実験用モデルに切り替えて続行する(どちらを使ったかはreport_numbers.jsonに残す)。
PROMPT_MODEL_USED = STRONG_MODEL
try:
    optimized = compile_planner(strong_lm)
except Exception as e:
    print(f"\nMIPROv2 が {STRONG_MODEL} で失敗しました: {type(e).__name__}: {str(e)[:300]}")
    print(f"-> 指示文の提案モデルを {MODEL} に切り替えて再実行します (再試行 {adapter.n_retry} 回)\n")
    PROMPT_MODEL_USED = MODEL
    optimized = compile_planner(lm)
print("指示文の提案に使ったモデル:", PROMPT_MODEL_USED)
optimized.save("optimized_planner.json")
check_budget()


## 9. 最適化後の成功率と、学習された内容

In [ ]:
opt_res = evaluate(optimized)
print(f"baseline {base_res.score:.1f}%  ->  optimized {opt_res.score:.1f}%")

print("\n=== 最適化後の指示文 ===")
print(optimized.plan.predict.signature.instructions)
print(f"\n=== few-shot 例: {len(optimized.plan.predict.demos)} 件 ===")
for d in optimized.plan.predict.demos[:1]:
    print(d.get("instruction"), "\n", d.get("plan"), "\n---")
check_budget()

In [ ]:
import pandas as pd

def by_kind(res):
    rows = [(TASK_BY_SEED[ex.seed]["kind"], float(score)) for ex, _, score in res.results]
    return pd.DataFrame(rows, columns=["kind", "ok"]).groupby("kind")["ok"].agg(["mean", "count"])

table = pd.concat({"baseline": by_kind(base_res)["mean"], "optimized": by_kind(opt_res)["mean"]}, axis=1)
table["n"] = by_kind(base_res)["count"]
table.loc["ALL"] = [base_res.score / 100, opt_res.score / 100, len(testset)]
display((table[["baseline", "optimized"]] * 100).round(1).assign(n=table["n"].astype(int)))
table.to_csv("results_by_kind.csv")

# 失敗の理由 (ベースライン): 何のルールで落ちているか
from collections import Counter
def reasons(res):
    c = Counter()
    for ex, pred, sc in res.results:
        if not sc:
            ok, err, _ = run_task(TASK_BY_SEED[ex.seed], getattr(pred, "plan", None) or "", controller="fast")
            c[re.sub(r"bag\d+", "bagN", re.sub(r"\d+", "N", err))[:70] or "(goal not reached)"] += 1
    return c.most_common(6)
print("baseline の失敗理由:", reasons(base_res))
print("optimized の失敗理由:", reasons(opt_res))

## 9.5 対照実験: 手書きプロンプトとの比較
DSPyの最適化に意味があったかを確かめるため、**人が書いた**プロンプトを同じ48件で測ります。
- 手書き(指示文): 出力形式(1行に1スキル、番号や説明文なし)だけを書く。積付ルールは書かない(DSPyが作った指示文にもルールは入っていないため、条件をそろえる)。
- 手書き(例): `train` の各種類(stack/mixed/dense/tight)から先頭の1件ずつ、計4件の正解プランを例として付ける(DSPyが選んだ例の数と同じ)。
DSPyの最適化後が、この手書きより高ければ、「探索に価値があった」と言えます。同程度なら、効いたのは「例と形式を示したこと」で、DSPyは必須ではありません。

In [ ]:
class PlanHand(dspy.Signature):
    """Write a loading plan for a robot arm. Output only skill calls, exactly one per line, with no numbering, no comments and no explanation.
    The instruction may contain a request about the order of handling (in English or Japanese); follow it.
    Skills: pick(bag), place_slot(slot_number), place_on(bag). A bag must be picked before it is placed."""
    instruction: str = dspy.InputField()
    scene: str = dspy.InputField(desc="attributes and locations of the bags and the container")
    plan: str = dspy.OutputField(desc="one skill call per line, e.g. pick(bag01) then place_slot(1)")


hand = Planner()
hand.plan = dspy.ChainOfThought(PlanHand)
demos, seen = [], set()
for ex in trainset:                                    # 各種類の先頭1件に、正解プラン(ソルバーの出力)を例として付ける
    k = TASK_BY_SEED[ex.seed]["kind"]
    if k in seen:
        continue
    seen.add(k)
    demos.append(dspy.Example(instruction=ex.instruction, scene=ex.scene,
                              reasoning="Fill the floor slots first, then stack the remaining bags on bags that are on the floor.",
                              plan=oracle_plan(TASK_BY_SEED[ex.seed])))
hand.plan.predict.demos = demos
print("手書きの例:", len(demos), "件 (種類:", sorted(seen), ")")

hand_res = evaluate(hand)
print(f"baseline {base_res.score:.1f}%  |  手書き(形式+例4件) {hand_res.score:.1f}%  |  DSPy 最適化後 {opt_res.score:.1f}%   (n={len(testset)})")
table_h = pd.concat({"baseline": by_kind(base_res)["mean"], "handwritten": by_kind(hand_res)["mean"], "optimized": by_kind(opt_res)["mean"]}, axis=1)
table_h.loc["ALL"] = [base_res.score / 100, hand_res.score / 100, opt_res.score / 100]
print((table_h * 100).round(1))
print("handwritten の失敗理由:", reasons(hand_res))


def by_order(res):
    rows = [(str(TASK_BY_SEED[ex.seed].get("order")), float(score)) for ex, _, score in res.results]
    return pd.DataFrame(rows, columns=["order", "ok"]).groupby("order")["ok"].agg(["mean", "count"])

table_o = pd.concat({"baseline": by_order(base_res)["mean"], "handwritten": by_order(hand_res)["mean"], "optimized": by_order(opt_res)["mean"]}, axis=1)
table_o["n"] = by_order(base_res)["count"]
print("指示文の順序の要求ごとの成功率 (None = 要求なし):")
display((table_o[["baseline", "handwritten", "optimized"]] * 100).round(1).assign(n=table_o["n"].astype(int)))
check_budget()

## 10. 上位と下位をつなぐ(プランナーの計画を、学習した方策で実行する)
最適化したプランナーが出した計画を、**GPUで学習した下位方策**で実際に動かし、ミッションとしての成功率を測ります。
熟練者コントローラの場合と比べると、下位方策の影響が分かります(動作ノイズ 0.6cm は外乱の模擬。BCはラウンド 0 の方策、learnedはDAgger後の方策)。

In [ ]:
def end_to_end(res, controller, noise=0.0, limit=None):
    rows = res.results[:limit] if limit else res.results
    ok = 0
    for ex, pred, _ in rows:
        o, _, _ = run_task(TASK_BY_SEED[ex.seed], getattr(pred, "plan", None) or "", controller=controller, act_noise=noise)
        ok += o
    return ok / len(rows)

t0 = time.time()
E2E_N = 24     # 時間の目安: 1 ミッションあたり数秒
e2e = {
    "optimized plan + fast(計画だけ)": opt_res.score / 100,
    "optimized plan + expert": end_to_end(opt_res, expert_action, 0.0, E2E_N),
    "optimized plan + learned policy": end_to_end(opt_res, ctrl, 0.0, E2E_N),
    "optimized plan + expert (noise)": end_to_end(opt_res, expert_action, 0.006, E2E_N),
    "optimized plan + BC policy (noise)": end_to_end(opt_res, bc_ctrl, 0.006, E2E_N),
    "optimized plan + learned policy (noise)": end_to_end(opt_res, ctrl, 0.006, E2E_N),
}
for k, v in e2e.items():
    print(f"{k:42s} {v:.0%}")
print(f"({(time.time() - t0) / 60:.1f} 分)")

## 11. (任意) 失敗時のやり直し(Refine)
`dspy.Refine`で、失敗したら最大3回やり直します。呼び出しが増えるので、費用の目安は`check_budget()`で確認してください。
(強いモデルでの比較は、応答が空になる問題が解決しなかったため、このノートブックでは行いません。)

In [ ]:
SEED_OF = {(e.instruction, e.scene): e.seed for e in testset}

def reward(args, pred):
    ok, _, _ = run_task(TASK_BY_SEED[SEED_OF[(args["instruction"], args["scene"])]], getattr(pred, "plan", None) or "", controller="fast")
    return float(ok)

refined = dspy.Refine(module=optimized, N=3, reward_fn=reward, threshold=1.0)
ref_res = evaluate(refined)
print(f"baseline {base_res.score:.1f}%  optimized {opt_res.score:.1f}%  optimized+Refine {ref_res.score:.1f}%")
check_budget()

## 11.5 Genesisで強化学習と模倣学習を比べる
荷物を1個つかみ、スロットに置く動作を、**同じ環境・同じノイズ・同じ評価**で、熟練者のプログラム、BC、DAgger、PPO(強化学習)の 4 通りで比べます。
- 環境: Genesis(GPUで2048環境を並列に動かす物理シミュレーター)。手先(ガントリー)をxyzに動かし、吸着で荷物を持ち、離して置く。置き位置の許容誤差は8mm、動作ノイズは0.6cmで、MuJoCoの設計に合わせています。環境と課題は別物なので、MuJoCo側の数字とは並べません。
- 本体のMuJoCo・PyTorchと干渉しないように、**別の仮想環境のサブプロセス**で動かします(Modalのイメージには`/opt/gvenv`として作ってあります。無ければその場で作ります)。環境変数`SKIP_GENESIS=1`で飛ばせます。失敗しても、この先の動画づくりは続けられます(Genesisの場面だけが省かれます)。
- 所要時間の目安は15〜25分です。

`run_v7_on_modal.py` の既定では、この章の計算は別のGPUコンテナで先に始まっていて(`run_genesis`)、ここではその結果を受け取るだけです(環境変数 `GENESIS_CALL_ID`)。待ち時間の上限(既定 90 分)を過ぎたときや失敗したときは、Genesis の場面なしで先に進みます。`--no-parallel-genesis` を付けると、このコンテナの中で順に実行します。


In [ ]:
%%writefile genesis_part.py
"""Genesis上で、荷物を1個つかんでスロットに置く動作をPPOで強化学習する。

MuJoCo版(DAgger)と同じ設計にそろえてある:
  - 手先(ガントリー)をxyzに動かし、吸着で荷物を持ち、離して置く
  - 置き位置の許容誤差8mm、動作ノイズ0.6cm
  - 荷物は剛体で、離したあとの落下・接触はGenesisが計算する
"""
import math, time, json
import numpy as np
import torch
import torch.nn as nn
import genesis as gs

TOL = 0.008            # 置き位置の許容誤差 [m]
ACT_NOISE = 0.006      # 動作ノイズの標準偏差 [m / ステップ]
STEP_MAX = 0.02        # 1ステップに手先が動ける最大量 [m]
EP_LEN = 150
CUBE = 0.05
OBS_DIM, ACT_DIM = 11, 4


class PickPlaceEnv:
    def __init__(self, n_envs, device="cuda", seed=0, render=False):
        self.n, self.dev = n_envs, torch.device(device)
        self.g = torch.Generator(device=self.dev); self.g.manual_seed(seed)
        self.scene = gs.Scene(show_viewer=False, sim_options=gs.options.SimOptions(dt=0.02, substeps=2))
        self.scene.add_entity(gs.morphs.Plane())
        self.render = render
        col = (lambda rgb: dict(surface=gs.surfaces.Default(color=rgb))) if render else (lambda rgb: {})
        self.cube = self.scene.add_entity(gs.morphs.Box(size=(CUBE, CUBE, CUBE), pos=(0.5, 0.0, CUBE / 2)), **col((0.2, 0.4, 0.9)))
        if render:                                                    # 動画用: 手先 (赤い球)、置き場所 (緑の板)、カメラ。物理には影響しない
            self.ee_vis = self.scene.add_entity(gs.morphs.Sphere(radius=0.02, pos=(0.5, 0.0, 0.25), collision=False), **col((0.9, 0.2, 0.15)))
            self.tg_vis = self.scene.add_entity(gs.morphs.Box(size=(0.06, 0.06, 0.002), pos=(0.5, 0.2, 0.001), collision=False), **col((0.2, 0.8, 0.3)))
            self.cam = self.scene.add_camera(res=(640, 480), pos=(1.15, -0.95, 0.75), lookat=(0.5, 0.0, 0.05), fov=40, GUI=False)
        self.scene.build(n_envs=n_envs)
        self.lo = torch.tensor([0.30, -0.35, 0.03], device=self.dev)
        self.hi = torch.tensor([0.70, 0.35, 0.35], device=self.dev)
        self.ee = torch.zeros(n_envs, 3, device=self.dev)
        self.target = torch.zeros(n_envs, 3, device=self.dev)
        self.attached = torch.zeros(n_envs, dtype=torch.bool, device=self.dev)
        self.picked = torch.zeros(n_envs, dtype=torch.bool, device=self.dev)
        self.t = 0
        self.reset()

    def _u(self, n, a, b):
        return a + (b - a) * torch.rand(n, device=self.dev, generator=self.g)

    def reset(self):
        n = self.n
        cx, cy = self._u(n, 0.38, 0.62), self._u(n, -0.25, 0.25)
        tx, ty = self._u(n, 0.38, 0.62), self._u(n, -0.30, 0.30)
        far = torch.hypot(tx - cx, ty - cy) > 0.12                       # 荷物とスロットは離す
        tx = torch.where(far, tx, cx + 0.15 * torch.sign(0.5 - cx + 1e-6))
        ty = torch.where(far, ty, cy)
        self.target = torch.stack([tx, ty, torch.full_like(tx, CUBE / 2)], 1)
        pos = torch.stack([cx, cy, torch.full_like(cx, CUBE / 2)], 1)
        idx = torch.arange(n, device=self.dev)
        self.cube.set_pos(pos, envs_idx=idx, zero_velocity=True)
        self.cube.set_quat(torch.tensor([1.0, 0, 0, 0], device=self.dev).repeat(n, 1), envs_idx=idx, zero_velocity=True)
        self.ee = torch.stack([self._u(n, 0.45, 0.55), self._u(n, -0.05, 0.05), torch.full((n,), 0.25, device=self.dev)], 1)
        self.attached[:] = False; self.picked[:] = False
        self.t = 0
        self.scene.step()
        return self.obs()

    def draw(self):
        """動画用: 1 フレーム (480, 640, 3) の画像を返す。"""
        self.ee_vis.set_pos(self.ee, zero_velocity=True)
        self.tg_vis.set_pos(self.target * torch.tensor([1.0, 1.0, 0.0], device=self.dev) + torch.tensor([0.0, 0.0, 0.001], device=self.dev), zero_velocity=True)
        out = self.cam.render()
        rgb = np.asarray(out[0] if isinstance(out, (tuple, list)) else out)
        while rgb.ndim > 3:
            rgb = rgb[0]
        return rgb.astype(np.uint8)

    def cube_pos(self):
        return self.cube.get_pos().reshape(self.n, 3)

    def obs(self):
        c = self.cube_pos()
        # 8mmの違いがMLPに見えるように、相対位置を20倍に拡大した特徴を使う
        return torch.cat([self.ee, 20.0 * (c - self.ee), 20.0 * (self.target - c), self.attached.float()[:, None],
                          torch.full((self.n, 1), self.t / EP_LEN, device=self.dev)], 1)

    def step(self, a, act_noise=ACT_NOISE):
        a = a.clamp(-1, 1)
        delta = a[:, :3] * STEP_MAX
        if act_noise > 0:
            delta = delta + act_noise * torch.randn(self.n, 3, device=self.dev, generator=self.g)
        self.ee = torch.max(torch.min(self.ee + delta, self.hi), self.lo)
        cmd = a[:, 3] > 0
        c = self.cube_pos()
        near = (torch.hypot(self.ee[:, 0] - c[:, 0], self.ee[:, 1] - c[:, 1]) < 0.02) & \
               ((self.ee[:, 2] - c[:, 2]) > 0.0) & ((self.ee[:, 2] - c[:, 2]) < 0.05)
        newly = cmd & near & ~self.attached
        self.attached = (self.attached | newly) & cmd
        self.picked |= self.attached
        idx = torch.nonzero(self.attached).reshape(-1)
        if idx.numel() > 0:                                               # 吸着中は、荷物を手先に追従させる(MuJoCo版と同じ簡易吸着)
            hold = torch.stack([self.ee[idx, 0], self.ee[idx, 1], self.ee[idx, 2] - 0.03], 1)
            self.cube.set_pos(hold, envs_idx=idx, zero_velocity=True)
        self.scene.step()
        self.t += 1
        c = self.cube_pos()
        d_xy = torch.hypot(c[:, 0] - self.target[:, 0], c[:, 1] - self.target[:, 1])
        d_reach = torch.linalg.norm(self.ee - (c + torch.tensor([0, 0, 0.04], device=self.dev)), dim=1)
        # 報酬: 持ち上げる前は「荷物に近いほど」、持ち上げたあとは「スロットに近いほど」大きい。
        # 持ち上げた状態の報酬は、持ち上げる前の最大値(1.0)より必ず大きくして、その場にとどまる局所解を避ける。
        reach = 1.0 - torch.tanh(5.0 * d_reach)
        carry = 1.0 + (1.0 - torch.tanh(5.0 * d_xy)) + 2.0 * (1.0 - torch.tanh(50.0 * d_xy))
        r = torch.where(self.picked, carry, reach) + 0.5 * newly.float()
        success = (~self.attached) & (d_xy < TOL) & (c[:, 2] < CUBE / 2 + 0.01) & self.picked
        r = r + 3.0 * success.float()
        done = self.t >= EP_LEN
        return self.obs(), r, done, success


class AC(nn.Module):
    def __init__(self, h=256):
        super().__init__()
        mk = lambda o: nn.Sequential(nn.Linear(OBS_DIM, h), nn.ELU(), nn.Linear(h, h), nn.ELU(), nn.Linear(h, o))
        self.pi, self.v = mk(ACT_DIM), mk(1)
        self.log_std = nn.Parameter(torch.full((ACT_DIM,), -1.0))

    def dist(self, o):
        return torch.distributions.Normal(self.pi(o), self.log_std.exp())


@torch.no_grad()
def evaluate(env, ac, act_noise, episodes=1):
    ok = []
    for _ in range(episodes):
        o = env.reset()
        for _ in range(EP_LEN):
            o, r, done, s = env.step(ac.pi(o), act_noise)
        ok.append(s.float().mean().item())
    return float(np.mean(ok))


def train(n_envs=2048, iters=150, act_noise=ACT_NOISE, seed=0, log=print, env=None):
    torch.manual_seed(seed)
    env = env or PickPlaceEnv(n_envs, seed=seed)
    dev = env.dev
    ac = AC().to(dev)
    opt = torch.optim.Adam(ac.parameters(), lr=3e-4)
    hist = []
    best = (-1.0, None, -1)                                          # (学習中の成功率, 重み, イテレーション)
    t0 = time.time()
    for it in range(iters):
        for gparam in opt.param_groups:                              # 学習率を3e-4から3e-5まで線形に下げる (後半の崩れを防ぐ)
            gparam["lr"] = 3e-4 * (1.0 - 0.9 * it / max(iters - 1, 1))
        O = torch.zeros(EP_LEN, env.n, OBS_DIM, device=dev); A = torch.zeros(EP_LEN, env.n, ACT_DIM, device=dev)
        LP = torch.zeros(EP_LEN, env.n, device=dev); R = torch.zeros(EP_LEN, env.n, device=dev); V = torch.zeros(EP_LEN + 1, env.n, device=dev)
        o = env.reset()
        for t in range(EP_LEN):
            with torch.no_grad():
                d = ac.dist(o); a = d.sample()
                O[t], A[t], LP[t], V[t] = o, a, d.log_prob(a).sum(-1), ac.v(o).squeeze(-1)
            o, r, done, succ = env.step(a, act_noise)
            R[t] = r
        train_succ = succ.float().mean().item()
        c_end = env.cube_pos()
        d_end = torch.hypot(c_end[:, 0] - env.target[:, 0], c_end[:, 1] - env.target[:, 1])
        picked_rate, d_med = env.picked.float().mean().item(), d_end.median().item()
        adv = torch.zeros_like(R); last = 0
        for t in reversed(range(EP_LEN)):
            nv = V[t + 1] if t + 1 < EP_LEN else 0.0
            delta = R[t] + 0.99 * nv - V[t]
            last = delta + 0.99 * 0.95 * last
            adv[t] = last
        ret = adv + V[:EP_LEN]
        Of, Af, LPf, advf, retf = O.reshape(-1, OBS_DIM), A.reshape(-1, ACT_DIM), LP.reshape(-1), adv.reshape(-1), ret.reshape(-1)
        advf = (advf - advf.mean()) / (advf.std() + 1e-8)
        N = Of.shape[0]
        for _ in range(5):
            perm = torch.randperm(N, device=dev)
            for mb in perm.chunk(8):
                d = ac.dist(Of[mb]); lp = d.log_prob(Af[mb]).sum(-1)
                ratio = (lp - LPf[mb]).exp()
                pl = -torch.min(ratio * advf[mb], ratio.clamp(0.8, 1.2) * advf[mb]).mean()
                vl = ((ac.v(Of[mb]).squeeze(-1) - retf[mb]) ** 2).mean()
                opt.zero_grad(); (pl + 0.5 * vl).backward(); nn.utils.clip_grad_norm_(ac.parameters(), 1.0); opt.step()
        if train_succ > best[0]:
            best = (train_succ, {k: v.detach().clone() for k, v in ac.state_dict().items()}, it)
        hist.append(dict(iter=it, train_success=train_succ, picked=picked_rate, d_med=d_med, reward=R.sum(0).mean().item(), sec=time.time() - t0))
        if it % 5 == 0 or it == iters - 1:
            log(f"iter {it:3d} | 成功率 {train_succ:.1%} | 持ち上げ {picked_rate:.0%} | 最終距離(中央値) {d_med*100:.1f}cm | 報酬 {hist[-1]['reward']:.2f} | {hist[-1]['sec']:.0f}s")
    if best[1] is not None:                                          # 学習中の成功率が最良だったときの重みを返す
        ac.load_state_dict(best[1]); log(f"最良のイテレーション: {best[2]} (学習中の成功率 {best[0]:.1%})")
    return ac, hist, env




# ---------------------------------------------------------------------------------------------
# 熟練者 (ルールを知っているプログラム)、BC、DAgger。PPOと同じ環境・同じノイズで比べる。
# ---------------------------------------------------------------------------------------------
@torch.no_grad()
def expert_action(env):
    """荷物の上まで移動 -> 下りて吸着 -> 持ち上げ -> スロットの上まで移動 -> 下りて、真上に来たら離す。"""
    c, ee, tg = env.cube_pos(), env.ee, env.target
    dev = env.dev
    xy_c = torch.hypot(ee[:, 0] - c[:, 0], ee[:, 1] - c[:, 1])
    xy_t = torch.hypot(c[:, 0] - tg[:, 0], c[:, 1] - tg[:, 1])
    cruise = 0.12
    # 吸着前
    des_pick = torch.stack([c[:, 0], c[:, 1], torch.where(xy_c > 0.010, torch.full_like(xy_c, cruise), c[:, 2] + 0.03)], 1)
    suck = (xy_c < 0.012) & ((ee[:, 2] - c[:, 2]) < 0.045)
    # 吸着中 (荷物は手先の3cm下)
    low = torch.where(xy_t > 0.010, torch.full_like(xy_t, cruise), torch.full_like(xy_t, 0.06))
    des_carry = torch.stack([tg[:, 0], tg[:, 1], low], 1)
    release = (xy_t < 0.005) & (ee[:, 2] < 0.07)
    carry_cmd = ~release
    # 離したあと
    des_done = ee.clone()
    des = torch.where(env.attached[:, None], des_carry, torch.where(env.picked[:, None], des_done, des_pick))
    cmd = torch.where(env.attached, carry_cmd, torch.where(env.picked, torch.zeros_like(suck), suck))
    a = ((des - ee) / STEP_MAX).clamp(-1, 1)
    return torch.cat([a, (cmd.float() * 2 - 1)[:, None]], 1)


class MLP(nn.Module):
    def __init__(self, h=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(OBS_DIM, h), nn.ELU(), nn.Linear(h, h), nn.ELU(), nn.Linear(h, ACT_DIM))

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def rollout(env, learner, beta, act_noise, collect):
    """learner: MLPかNone。betaの確率で熟練者の動作を実行(DAggerの混合)。collectなら(観測, 熟練者の動作)も返す。"""
    O, E = [], []
    o = env.reset()
    for _ in range(EP_LEN):
        ea = expert_action(env)
        if collect:
            O.append(o); E.append(ea)
        if learner is None:
            a = ea
        else:
            a = learner(o)
            if beta > 0:
                use = (torch.rand(env.n, 1, device=env.dev, generator=env.g) < beta)
                a = torch.where(use, ea, a)
        o, r, done, succ = env.step(a, act_noise)
    out = succ.float().mean().item()
    if collect:
        return out, torch.cat(O), torch.cat(E)
    return out


def fit(model, X, Y, epochs=30, lr=1e-3, bs=8192):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = X.shape[0]
    for ep in range(epochs):
        for g_ in opt.param_groups:
            g_["lr"] = lr * (1 - 0.9 * ep / max(epochs - 1, 1))
        perm = torch.randperm(n, device=X.device)
        for mb in perm.chunk(max(n // bs, 1)):
            loss = ((model(X[mb]) - Y[mb]) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return model


def evaluate_ctrl(env, learner, act_noise, seed=12345):
    env.g.manual_seed(seed)                                           # 同じ荷物・スロット・ノイズで比べる
    return rollout(env, learner, 0.0, act_noise, collect=False)


def run_imitation(env, rounds=4, log=print):
    res = {}
    res["expert_clean"] = evaluate_ctrl(env, None, 0.0)
    res["expert_noise"] = evaluate_ctrl(env, None, ACT_NOISE)
    log(f"熟練者: ノイズなし {res['expert_clean']:.1%} / ノイズ {res['expert_noise']:.1%}")
    _, X, Y = rollout(env, None, 1.0, 0.0, collect=True)               # 熟練者のデモ (ノイズなし)
    model = bc_model = fit(MLP().to(env.dev), X, Y)
    res["bc_clean"], res["bc_noise"] = evaluate_ctrl(env, model, 0.0), evaluate_ctrl(env, model, ACT_NOISE)
    log(f"BC (デモ {X.shape[0]}): ノイズなし {res['bc_clean']:.1%} / ノイズ {res['bc_noise']:.1%}")
    res["dagger"] = []
    for r in range(1, rounds + 1):
        _, X2, Y2 = rollout(env, model, max(0.4 - 0.2 * r, 0.0), ACT_NOISE, collect=True)   # 評価と同じノイズの下で、学習した方策を動かす
        X, Y = torch.cat([X, X2]), torch.cat([Y, Y2])
        model = fit(MLP().to(env.dev), X, Y)
        row = dict(round=r, samples=int(X.shape[0]), clean=evaluate_ctrl(env, model, 0.0), noise=evaluate_ctrl(env, model, ACT_NOISE))
        res["dagger"].append(row)
        log(f"DAgger ラウンド {r} (データ {row['samples']}): ノイズなし {row['clean']:.1%} / ノイズ {row['noise']:.1%}")
    return res, model, bc_model


def run_all(n_envs=2048, ppo_iters=250, log=print):
    env = PickPlaceEnv(n_envs, seed=0)
    out = {}
    t0 = time.time()
    out["imitation"], dag_model, bc_model = run_imitation(env, 4, log)
    log(f"(模倣学習 {time.time() - t0:.0f} 秒)")
    t0 = time.time()
    ac, hist, env = train(n_envs=n_envs, iters=ppo_iters, env=env, log=log)
    out["ppo"] = dict(clean=evaluate(env, ac, 0.0), noise=evaluate(env, ac, ACT_NOISE), iters=ppo_iters, sec=time.time() - t0,
                      curve=[(h["iter"], h["train_success"]) for h in hist])
    log(f"PPO ({ppo_iters} イテレーション, {out['ppo']['sec']:.0f} 秒): ノイズなし {out['ppo']['clean']:.1%} / ノイズ {out['ppo']['noise']:.1%}")
    return out, ac, dag_model, bc_model, env


@torch.no_grad()
def run_episode(env, act_fn, seed, record_every=0):
    env.g.manual_seed(seed)                                           # 同じ荷物・スロット・ノイズ列で比べる
    o = env.reset()
    frames = [env.draw()] if record_every else []
    for t in range(EP_LEN):
        o, r, done, s_ = env.step(act_fn(o), ACT_NOISE)
        if record_every and (t + 1) % record_every == 0:
            frames.append(env.draw())
    return bool(s_[0]), frames


def make_clips(ac, dag_model, bc_model, path="genesis_clips.mp4", every=3, log=print):
    """BC / DAgger / PPO の3つを、同じ荷物・同じノイズで1エピソードずつ描画する。BCが失敗し、DAggerとPPOが成功するseedを探す。"""
    import mediapy
    env = PickPlaceEnv(1, seed=0, render=True)
    pol = [("BC only", lambda o: bc_model(o)), ("DAgger", lambda o: dag_model(o)), ("PPO (RL)", lambda o: ac.pi(o))]
    seed = 700
    for k in range(700, 730):
        ok = [run_episode(env, f, k)[0] for _, f in pol]
        if (not ok[0]) and ok[1] and ok[2]:
            seed = k; break
    log(f"動画用の seed: {seed}")
    allf, info = [], []
    for name, f in pol:
        ok, fr = run_episode(env, f, seed, record_every=every)
        allf += fr; info.append(dict(name=name, frames=len(fr), success=ok))
        log(f"  {name}: {'成功' if ok else '失敗'} ({len(fr)} フレーム)")
    mediapy.write_video(path, np.stack(allf), fps=20)
    return dict(seed=seed, items=info, every=every)


def save_chart(out, path="genesis_compare.png"):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    im, pp = out["imitation"], out["ppo"]
    names = ["Expert (program)", "BC only"] + [f"DAgger round {d['round']}" for d in im["dagger"]] + ["PPO (RL)"]
    vals = [im["expert_noise"], im["bc_noise"]] + [d["noise"] for d in im["dagger"]] + [pp["noise"]]
    fig, ax = plt.subplots(figsize=(6.4, 4.8), dpi=100)
    cols = ["#888888", "#d9822b"] + ["#3b6fb6"] * len(im["dagger"]) + ["#2e9e5b"]
    ax.barh(names[::-1], [v * 100 for v in vals][::-1], color=cols[::-1])
    for i, v in enumerate(vals[::-1]):
        ax.text(v * 100 + 1, i, f"{v * 100:.0f}%", va="center", fontsize=9)
    ax.set_xlim(0, 112); ax.set_xlabel("success rate (%) with 0.6cm noise, 8mm tolerance")
    ax.set_title("Genesis: imitation vs reinforcement learning")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def main():
    import sys, traceback
    n_envs = int(sys.argv[1]) if len(sys.argv) > 1 else 2048
    iters = int(sys.argv[2]) if len(sys.argv) > 2 else 250
    gs.init(backend=gs.gpu, logging_level="warning")
    out, ac, dag_model, bc_model, env = run_all(n_envs, iters, log=lambda m: print(m, flush=True))
    save_chart(out)
    res = dict(env=dict(n_envs=n_envs, noise_m=ACT_NOISE, tol_m=TOL, ep_len=EP_LEN), imitation=out["imitation"],
               ppo={k: v for k, v in out["ppo"].items() if k != "curve"}, ppo_curve_every10=out["ppo"]["curve"][::10],
               genesis=getattr(gs, "__version__", "?"), torch=torch.__version__, device=torch.cuda.get_device_name(0), clips=None)
    json.dump(res, open("genesis_results.json", "w"), ensure_ascii=False, indent=1)
    torch.save(ac.state_dict(), "genesis_ppo_policy.pt"); torch.save(dag_model.state_dict(), "genesis_dagger_policy.pt")
    try:                                                              # 描画に失敗しても、数値の結果は残す
        res["clips"] = make_clips(ac, dag_model, bc_model, log=lambda m: print(m, flush=True))
    except Exception:
        traceback.print_exc(); print("描画に失敗したため、動画用のクリップは作りません (数値の結果は保存済み)", flush=True)
    json.dump(res, open("genesis_results.json", "w"), ensure_ascii=False, indent=1)
    print("DONE", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
import subprocess, sys, os, json, time
t0 = time.time()
GEN = None
GENESIS_FILES = ("genesis_results.json", "genesis_clips.mp4", "genesis_compare.png")   # 12章(動画)と13章が読むファイル


def _genesis_python():
    """Genesis 用の Python (別 venv)。未インストールなら作る (再実行時は飛ばす)"""
    PY = next((p for p in ("/opt/gvenv/bin/python", os.path.expanduser("~/gvenv/bin/python"))
               if os.path.exists(p) and subprocess.run([p, "-c", "import genesis"], capture_output=True).returncode == 0), None)
    if PY is None:
        PY = os.path.expanduser("~/gvenv/bin/python")
        if subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", os.path.dirname(os.path.dirname(PY))]).returncode != 0:
            subprocess.run([sys.executable, "-m", "pip", "-q", "install", "virtualenv"], check=True)                # venvが使えない環境への保険
            subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", os.path.dirname(os.path.dirname(PY))], check=True)
        subprocess.run([PY, "-m", "pip", "-q", "install", "genesis-world"], check=True)
    subprocess.run([PY, "-c", "import genesis; print('genesis', genesis.__version__)"], check=True)
    return PY


def _run_genesis_here():
    """このコンテナで genesis_part.py を実行する (従来どおり)"""
    PY = _genesis_python()
    p = subprocess.Popen([PY, "-u", "genesis_part.py", "2048", "250"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=dict(os.environ, PYOPENGL_PLATFORM="egl"))
    for line in p.stdout:
        if line.strip() and not any(k in line for k in ("libEGL", "DeprecationWarning", "kwarg_node", "return index")):
            print(line, end="")
    p.wait()


def _wait_genesis_elsewhere(call_id):
    """run_v7_on_modal.py が別の GPU コンテナで先に走らせた Genesis (run_genesis) の結果を待ち、ファイルをここに置く"""
    import modal
    wait_s = float(os.environ.get("GENESIS_WAIT_MIN", "90")) * 60
    print(f"別コンテナの Genesis (call id {call_id}) を待ちます (上限 {wait_s / 60:.0f} 分)")
    res = modal.FunctionCall.from_id(call_id).get(timeout=wait_s)      # 上限を過ぎるとTimeoutError -> 下のexceptで「場面なし」に進む
    print(f"Genesis コンテナ: ok={res['ok']} returncode={res['returncode']} {res['minutes']:.1f} 分 | ログ末尾:\n{res['log_tail']}")
    for name, data in res["files"].items():
        with open(name, "wb") as f:
            f.write(data)


try:
    if os.environ.get("SKIP_GENESIS", "0") == "1":
        raise RuntimeError("SKIP_GENESIS=1 のため飛ばします")
    for f in GENESIS_FILES:
        if os.path.exists(f):
            os.remove(f)
    call_id = os.environ.get("GENESIS_CALL_ID", "")
    if call_id:
        _wait_genesis_elsewhere(call_id)
    else:
        _run_genesis_here()
    if os.path.exists("genesis_results.json"):
        GEN = json.load(open("genesis_results.json"))
except Exception as e:
    print("Genesis の部分が失敗しました (動画は Genesis の場面なしで作ります):", repr(e))
print(f"({(time.time() - t0) / 60:.1f} 分)", "| 結果:", "あり" if GEN else "なし")


In [ ]:
if GEN:
    import pandas as pd
    im, pp = GEN["imitation"], GEN["ppo"]
    rows = [("熟練者 (プログラム)", im["expert_clean"], im["expert_noise"]), ("BC (デモのみ)", im["bc_clean"], im["bc_noise"])]
    rows += [(f"DAgger ラウンド {d['round']}", d["clean"], d["noise"]) for d in im["dagger"]] + [("PPO (強化学習)", pp["clean"], pp["noise"])]
    display((pd.DataFrame(rows, columns=["方法", "ノイズなし", "ノイズ 0.6cm"]).set_index("方法") * 100).round(1))
    from IPython.display import Image
    display(Image("genesis_compare.png", width=480))

## 12. 動画を作る
提出する動画は**3分以内(2倍速まで可)**です。このセルは2倍速(2アクションごとに1フレームを保存し、20fpsで書き出す)で作ります。
構成: タイトル → 仕組み → 下位方策の比較(BC対DAgger) → ミッションごとに「最適化前の計画の失敗」と「最適化後の計画を学習済み方策が実行する全工程」→ Genesis(強化学習と模倣学習の比較)→ 下位方策の学習曲線 → 学習された指示文 → 成功率。
12個のミッション(`tight`)は2倍速でも1分を超えます。見積もりが `VIDEO_BUDGET_S` に収まる範囲で、複数のミッションを入れます。

In [ ]:
import cv2, textwrap
import matplotlib.pyplot as plt

VIDEO_BUDGET_S = 170          # 動画全体の目標 (秒)。提出の上限は180秒
FPS = 20
OUT_MP4 = "final_demo.mp4"

def banner(frame, head, cap, color):
    f = frame.copy()
    cv2.rectangle(f, (0, 0), (f.shape[1], 34), (20, 20, 20), -1)
    cv2.putText(f, head[:62], (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)
    if cap:
        cv2.putText(f, cap, (10, f.shape[0] - 14), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
    return f

def card(lines, seconds, size=0.8, width=44, top=150, step=40):
    img = np.full((480, 640, 3), 25, np.uint8)
    wrapped = [w for s in lines for w in (textwrap.wrap(s, width) or [""])]
    for i, s in enumerate(wrapped[: (480 - top) // step]):
        cv2.putText(img, s, (20, top + step * i), cv2.FONT_HERSHEY_SIMPLEX, size, (255, 255, 255), 2 if size > 0.6 else 1, cv2.LINE_AA)
    return img, int(seconds * FPS)

def image_card(path, seconds):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (640, 480)), int(seconds * FPS)

def chart_card(table, seconds):
    fig, ax = plt.subplots(figsize=(6.4, 4.8), dpi=100)
    t = table.drop(index="ALL", errors="ignore")
    x = np.arange(len(t)); ax.bar(x - 0.2, t["baseline"] * 100, 0.4, label="baseline"); ax.bar(x + 0.2, t["optimized"] * 100, 0.4, label="optimized")
    ax.set_xticks(x); ax.set_xticklabels(t.index); ax.set_ylim(0, 125); ax.set_ylabel("success rate (%)"); ax.legend(loc="upper left", ncol=2)
    ax.set_title("Planner success by mission kind (unseen test missions)")
    fig.tight_layout(); fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy(); plt.close(fig)
    return img, int(seconds * FPS)

by_seed_base = {ex.seed: pred for ex, pred, _ in base_res.results}
by_seed_opt = {ex.seed: (pred, sc) for ex, pred, sc in opt_res.results}
base_score = {ex.seed: sc for ex, _, sc in base_res.results}

# 候補: 最適化後に成功したミッション。ベースラインが失敗したものを先に、同じなら荷物の多い(長い)ものを先に使う
cands = [ex for ex in testset if by_seed_opt[ex.seed][1]]
cands.sort(key=lambda ex: (bool(base_score[ex.seed]), -TASK_BY_SEED[ex.seed]["n"]))
def dry_s(t, plan_text, c):
    """同じ条件で録画せずに実行し、2 倍速の動画になったときの長さ [秒] を返す (ノイズなしなので、録画時と同じ動きになる)。"""
    _, _, w = run_task(t, plan_text, controller=c)
    return w.data.time / 2 * 1.08
print("candidates:", [(ex.seed, TASK_BY_SEED[ex.seed]["kind"], "baseline FAIL" if not base_score[ex.seed] else "baseline ok") for ex in cands][:8])

GEN, GEN_CLIPS = None, None                                       # Genesisの結果(11.5章)。無ければ、Genesisの場面なしで作る
if os.path.exists("genesis_results.json"):
    GEN = json.load(open("genesis_results.json"))
    if GEN.get("clips") and os.path.exists("genesis_clips.mp4"):
        try:
            GEN_CLIPS = np.asarray(mediapy.read_video("genesis_clips.mp4"))
            if len(GEN_CLIPS) != sum(it["frames"] for it in GEN["clips"]["items"]):
                GEN_CLIPS = None
        except Exception as e:
            print("Genesis のクリップを読めません:", repr(e))
GEN_S = 0 if GEN is None else 4 + (0 if GEN_CLIPS is None else len(GEN_CLIPS) / FPS + 3) + 6
print("Genesis の場面:", "なし" if GEN is None else f"あり (約 {GEN_S:.0f} 秒, クリップ {'あり' if GEN_CLIPS is not None else 'なし'})")

writer = mediapy.VideoWriter(OUT_MP4, shape=(480, 640), fps=FPS, crf=24)
writer.__enter__()
n_frames = 0
def add(img, n=1):
    global n_frames
    for _ in range(n):
        writer.add_image(img)
    n_frames += n

def add_card(c):
    add(*c)

add_card(card(["Airport baggage loading", "LLM planner (DSPy)", "+ GPU-trained low-level policy"], 5, top=170, step=50))
add_card(card(["1. LLM writes a loading plan", "2. Planner prompt is optimized by", "    simulator success (DSPy MIPROv2)", "3. Neural policy (GPU, DAgger) moves", "    the arm: pick / place_slot / place_on", "Video: 2x speed"], 10, size=0.65, width=50, top=110, step=44))

REC.frame_every = 2          # 2アクションごとに1フレーム -> 20fpsで2倍速

# ---- 下位方策の比較: 同じミッションを、動作ノイズの下でBCとDAgger後の方策に実行させる ----
def _try(task, c, k):
    w = _cached_world(); w._rng = np.random.default_rng(k)
    return run_task(task, oracle_plan(task), controller=c, world=w, act_noise=0.006)[0]

pick_cmp = None
for s_ in range(5000, 5000 + 4 * 24, 4):                         # stackミッション(7～8個)の中から、BCが失敗しDAggerが成功するseedを探す
    tk = make_task(s_)
    if tk["kind"] == "stack" and not _try(tk, bc_ctrl, s_) and _try(tk, ctrl, s_):
        pick_cmp = (tk, s_); break
print("policy comparison mission:", None if pick_cmp is None else pick_cmp[1])
if pick_cmp:
    tk, ks = pick_cmp
    add_card(card(["Low-level policy, with 0.6cm noise", "Same mission, same plan:", "behavior cloning only vs DAgger", "(an example picked where BC failed;", " success rates over all missions later)"], 6, size=0.7, top=130, step=44))
    for name, c, col in (("BC only (no DAgger)", bc_ctrl, (255, 160, 120)), ("after DAgger", ctrl, (120, 255, 160))):
        REC.sink = lambda img, cap, name=name, col=col: add(banner(img, name, cap, col))
        REC._rng = np.random.default_rng(ks)
        ok, err, _ = run_task(tk, oracle_plan(tk), controller=c, world=REC, record=True, act_noise=0.006)
        add_card(card([f"{name}: {'SUCCESS' if ok else 'FAIL'}", err or "all bags loaded"], 3, size=0.75, top=190))
    REC.sink = None
    print(f"policy comparison done | video so far {n_frames / FPS:.0f}s")

shown = []
for ex in cands:
    t = TASK_BY_SEED[ex.seed]
    need = 4 + 3 + dry_s(t, getattr(by_seed_opt[ex.seed][0], "plan", None) or "", ctrl)
    if not base_score[ex.seed]:
        need += 3 + dry_s(t, getattr(by_seed_base[ex.seed], "plan", None) or "", ctrl)
    if n_frames / FPS + need + 26 + GEN_S > VIDEO_BUDGET_S:      # 末尾のカード(約25秒)とGenesisの場面を含めて、上限に収まらなければ飛ばす
        continue
    add_card(card([f"Mission: {t['instruction']}", f"{t['n']} bags, rules are NOT given to the planner"], 4))
    if not base_score[ex.seed]:                                  # 最適化前の失敗 (短い)
        REC.sink = lambda img, cap: add(banner(img, "BEFORE: baseline planner", cap, (255, 220, 120)))
        ok, err, _ = run_task(t, getattr(by_seed_base[ex.seed], "plan", None) or "", controller=ctrl, world=REC, record=True)
        add_card(card(["BEFORE: FAIL", err or "goal not reached"], 3, size=0.7, width=46, top=190))
    REC.sink = lambda img, cap: add(banner(img, "AFTER: optimized planner + learned policy", cap, (120, 255, 160)))
    ok, err, _ = run_task(t, getattr(by_seed_opt[ex.seed][0], "plan", None) or "", controller=ctrl, world=REC, record=True)
    add_card(card([f"AFTER: {'SUCCESS' if ok else 'FAIL'}", err or f"all {t['n']} bags loaded"], 3, size=0.8, top=190))
    shown.append((ex.seed, ok))
    print("recorded seed", ex.seed, "success:", ok, f"| video so far {n_frames / FPS:.0f}s")
REC.sink = None

if GEN is not None:                                                # Genesis: 模倣学習と強化学習の比較
    add_card(card(["Genesis (GPU-parallel physics)", "Same task, same 0.6cm noise:", "expert / BC / DAgger / PPO (RL)", "2048 environments at once"], 4, size=0.7, top=150, step=46))
    if GEN_CLIPS is not None:
        pos = 0
        for it, col in zip(GEN["clips"]["items"], ((255, 160, 120), (120, 255, 160), (160, 200, 255))):
            for fr in GEN_CLIPS[pos:pos + it["frames"]]:
                add(banner(cv2.resize(np.asarray(fr), (640, 480)), f"Genesis: {it['name']} (sim x{GEN['clips']['every'] * 0.02 * FPS:.1f})", "SUCCESS" if it["success"] else "FAIL", col))
            pos += it["frames"]
        add_card(card(["(clips: one example picked where", " BC failed; all rates in the next chart)"], 3, size=0.6, width=46, top=190))
    add_card(image_card("genesis_compare.png", 6))

add_card(image_card("dagger_curve.png", 5))
add_card(chart_card(table, 5))
add_card(card(["Learned prompt (excerpt):"] + textwrap.wrap(optimized.plan.predict.signature.instructions.replace("\n", " "), 66)[:9], 6, size=0.5, width=70, top=60, step=28))
add_card(card([f"Planner (DSPy): {base_res.score:.0f}% -> {opt_res.score:.0f}%  (n={len(testset)})",
               f"Policy with 0.6cm noise (n={EVAL_MISSIONS}):", f"  BC {history[0]['plan_noisy']:.0%} -> DAgger {history[-1]['plan_noisy']:.0%}",
               f"  (scripted expert {ref_noisy[0]:.0%})"], 9, size=0.7, top=150, step=46))
writer.__exit__(None, None, None)
print(f"video length: {n_frames / FPS:.0f} s  (提出の上限は 180 秒)")
if n_frames / FPS > 180:
    print("警告: 180 秒を超えています。VIDEO_BUDGET_S を下げて、このセルを再実行してください (方策の動作が遅いと、見積もりより長くなります)")
from IPython.display import Video, display
display(Video(OUT_MP4, embed=False, width=480))   # 埋め込まない (mp4は同じフォルダに保存され、ランナーがダウンロードする)

In [ ]:
import os
print("保存先:", os.path.abspath("final_demo.mp4"), f"({os.path.getsize('final_demo.mp4') / 1e6:.1f} MB)")
print("Modal では、実行後に run_v7_on_modal.py が modal_out/ へダウンロードします (手動なら: modal volume get pai-final-outputs <実行名> ./modal_out)")


## 13. レポート用の数値

In [ ]:
report = dict(
    models=dict(task=MODEL, prompt=PROMPT_MODEL_USED, prompt_planned=STRONG_MODEL, mipro_auto=MIPRO_AUTO, show_rules=SHOW_RULES),
    sizes=dict(train=N_TRAIN, dev=N_DEV, test=N_TEST, expert_episodes=N_EXPERT_EPISODES, dagger_rounds=DAGGER_ROUNDS, dagger_episodes=N_DAGGER_EPISODES, epochs=EPOCHS),
    device=DEVICE,
    policy_reference=dict(expert_clean=ref_clean, expert_noise=ref_noisy), eval_missions=EVAL_MISSIONS,
    policy_history=history,
    planner=dict(baseline=base_res.score, handwritten=hand_res.score, optimized=opt_res.score, by_kind={k: [float(v) for v in r] for k, r in table[["baseline", "optimized"]].iterrows()},
                 by_order={k: [float(v) for v in r] + [int(table_o.loc[k, "n"])] for k, r in table_o[["baseline", "handwritten", "optimized"]].iterrows()}),
    end_to_end=e2e,
    video_missions=shown,
    genesis=(None if GEN is None else {k: GEN[k] for k in ('env', 'imitation', 'ppo', 'ppo_curve_every10', 'genesis', 'device')}),
    llm_calls=dict(task=len(lm.history), strong=len(strong_lm.history), litellm_estimated_usd=round(spent_usd(), 2)),
)
with open("report_numbers.json", "w") as f:
    json.dump(report, f, ensure_ascii=False, indent=1, default=float)
print(json.dumps(report, ensure_ascii=False, indent=1, default=float))

In [ ]:
import glob
key = os.environ["ANTHROPIC_API_KEY"]
leaked = [f for f in glob.glob("*.json") + glob.glob("*.csv") + glob.glob("*.png") if key.encode() in open(f, "rb").read()]
assert not leaked, f"API キーが含まれているファイル: {leaked}"
print("OK: 保存ファイルに API キーは含まれていません。提出物:", glob.glob("*.json") + glob.glob("*.csv") + ["policy.pt", "final_demo.mp4 (YouTube へ)"])